# B.O.S.S. — Confronto YOLO11n vs YOLO11s su recordings

Confronto tra i due modelli YOLO11 (task s1-2, rivalutazione ADR-001) **a parità di condizioni**: stessi frame, stessa Ground Truth, stesse soglie, stessa risoluzione, stesso device. I modelli sono valutati in **istanti diversi** (uno dopo l'altro) ma con pipeline identica.

**Pipeline (per ciascun modello):**
1. Ricerca frame (dataset già estratto su Kaggle) — condivisa
2. Caricamento pesi + inferenza → frame annotati
3. Distribuzione classi e confidenza
4. Preparazione Ground Truth (remap nello spazio ID del modello)
5. Valutazione quantitativa contro GT (mAP, Precision, Recall)
6. Metriche per classe + grafici
7. Griglia campione frame + dashboard riepilogo
8. Report Markdown leggibile da IA

**Confronto finale (nuovo):** tabella metriche affiancate, barre raggruppate aggregate, AP@0.5 per classe, efficienza mAP@0.5 vs GMACs, dashboard e report `.md` di confronto.

---
**Output — `/kaggle/working/TEST_OUTPUT/` (ricreata da zero a ogni run):**
- `YOLO11n/` e `YOLO11s/` — una sottocartella per modello (`model_output/`, `gt_eval/`, `comparison/`, `markdown/`, `efficiency/`)
- `_confronto/` — tabella CSV, grafici e report Markdown di confronto tra i due modelli

---
**Kaggle — dati di input (già estratti sotto `/kaggle/input`):**
- **Dataset** recordings: cartelle `frames` / `frames(1)`
- **Dataset** GT: export Roboflow con `train/test/valid` + `data.yaml`
- **Model**: `yolo11n.pt` e `yolo11s.pt`

In [ ]:
import shutil, os
for entry in os.listdir("/kaggle/working"):
    p = os.path.join("/kaggle/working", entry)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("working pulita:", os.listdir("/kaggle/working"))


In [ ]:
# ============================================================
# Cella 1 — Import librerie
# ============================================================
%pip install ultralytics pyyaml tqdm torchmetrics onnx-tool --quiet

import os
import shutil
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import yaml

import matplotlib.pyplot as plt

from tqdm import tqdm
from ultralytics import YOLO

import torch
from torchmetrics.detection import MeanAveragePrecision

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cella 1b - Import modulo condiviso
# ============================================================
try:
    import boss_eval_utils as beu  # Kaggle: utility script auto-mounted in sys.path
except ModuleNotFoundError:
    import sys
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/lorenzoverdura/boss_eval_utils")
    import boss_eval_utils as beu
print(f"boss_eval_utils caricato da: {beu.__file__}")

In [ ]:
# ============================================================
# Cella 2 — Configurazione CONDIVISA (identica per i due modelli)
# ============================================================
# Condizioni di test IDENTICHE per entrambi i modelli: stessi frame, stessa GT,
# stesse soglie, stessa risoluzione, stesso device. Cambia solo il peso caricato.
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

if IS_KAGGLE:
    # ── Kaggle ──────────────────────────────────────────────
    KAGGLE_DATA_DIR  = Path("/kaggle/input/datasets/lorenzoverdura")
    KAGGLE_MODEL_DIR = Path("/kaggle/input/models/ultralytics/yolo11/pytorch/default/1")

    BASE_DIR       = Path("/kaggle/input")
    RECORDINGS_ZIP = KAGGLE_DATA_DIR  / "boss-recordings/images"
    GT_ZIP_PATH    = KAGGLE_DATA_DIR  / "boss-recordings1/BOSS_recordings.yolov11 (2)"
else:
    # ── Locale ──────────────────────────────────────────────
    BASE_DIR         = Path("/home/lorenzoverdura8/BOSS_CODE")
    KAGGLE_MODEL_DIR = BASE_DIR
    RECORDINGS_ZIP   = BASE_DIR / "recordings.zip"
    GT_ZIP_PATH      = BASE_DIR / "BOSS_recordings.yolov8 (2)/rf_split"

OUTPUT_DIR   = Path("/kaggle/working")
GT_EXTRACTED = OUTPUT_DIR / "ground_truth_boss"
TEST_OUTPUT  = OUTPUT_DIR / "TEST_OUTPUT"

# --- Classi e soglie dal modulo condiviso (no ridondanza) ---
BOSS_CLASSES   = beu.BOSS_CLASSES
NUM_CLASSES    = beu.NUM_CLASSES
CONF_THRESHOLD = beu.CONF_THRESHOLD   # 0.25
IOU_THRESHOLD  = beu.IOU_THRESHOLD    # 0.45 (NMS)
MATCH_IOU      = beu.MATCH_IOU        # 0.50 (TP/FP/FN)

IMG_SIZE = 640
DEVICE   = "0" if torch.cuda.is_available() else "cpu"


def resolve_weights(fname):
    """Risolve il path dei pesi: prima la dir modello montata, poi ricerca sotto
    /kaggle/input, infine nome nudo (Ultralytics scarica se c'e rete)."""
    cand = KAGGLE_MODEL_DIR / fname
    if cand.exists():
        return str(cand)
    if IS_KAGGLE:
        for p in Path("/kaggle/input").rglob(fname):
            return str(p)
    return fname


# I due modelli da confrontare a parita di condizioni (task s1-2, ADR-001)
MODELS = [
    {"name": "YOLO11n", "weights": resolve_weights("yolo11n.pt")},
    {"name": "YOLO11s", "weights": resolve_weights("yolo11s.pt")},
]

# TEST_OUTPUT ricreata UNA volta; ogni modello scrive in una sottocartella propria
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GT_EXTRACTED.mkdir(parents=True, exist_ok=True)
if TEST_OUTPUT.exists():
    shutil.rmtree(TEST_OUTPUT)
TEST_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Ambiente:       {'Kaggle' if IS_KAGGLE else 'Locale'}")
print(f"RECORDINGS_ZIP: {RECORDINGS_ZIP}  —  esiste: {RECORDINGS_ZIP.exists()}")
print(f"GT_ZIP_PATH:    {GT_ZIP_PATH}  —  esiste: {GT_ZIP_PATH.exists()}")
for m in MODELS:
    print(f"Modello {m['name']:8s}: {m['weights']}  —  esiste: {Path(m['weights']).exists()}")
print(f"CONF_THRESHOLD: {CONF_THRESHOLD} | IOU_THRESHOLD: {IOU_THRESHOLD} | MATCH_IOU: {MATCH_IOU}")
print(f"IMG_SIZE:       {IMG_SIZE} | DEVICE: {DEVICE}")

In [ ]:
# ============================================================
# Cella 3 — Ricerca frame (Kaggle: dataset già estratto)
# ============================================================
# Su Kaggle il dataset è già estratto in sola lettura sotto /kaggle/input,
# quindi NON si lavora con gli ZIP. RECORDINGS_ZIP punta direttamente alla
# cartella 'images' (le stesse immagini della GT, senza label). Le immagini
# vengono consolidate in un'unica cartella scrivibile in /kaggle/working (via
# symlink, senza copia), usata come sorgente per l'inferenza (Cella 4).

recordings_root = RECORDINGS_ZIP

# Fallback locale: se RECORDINGS_ZIP è un vero .zip, lo estrae in /kaggle/working.
if recordings_root.is_file() and zipfile.is_zipfile(recordings_root):
    extract_to = OUTPUT_DIR / "recordings"
    if not extract_to.exists() or not any(extract_to.iterdir()):
        print(f"Estrazione {recordings_root} ...")
        with zipfile.ZipFile(recordings_root, "r") as z:
            z.extractall(extract_to)
    recordings_root = extract_to

if not recordings_root.exists():
    raise FileNotFoundError(f"Cartella recordings non trovata: {recordings_root}")

# Cartella scrivibile che consolida i frame trovati sotto recordings_root.
# Symlink (no copia) per risparmiare tempo/disco.
TEST_FRAMES_DIR = OUTPUT_DIR / "frames_all"
if TEST_FRAMES_DIR.exists():
    shutil.rmtree(TEST_FRAMES_DIR)
TEST_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# Raccoglie ricorsivamente tutte le immagini sotto recordings_root.
img_files = sorted(
    p for p in recordings_root.rglob("*")
    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
if not img_files:
    raise FileNotFoundError(
        f"Nessuna immagine (.jpg/.jpeg/.png) trovata sotto {recordings_root}"
    )

all_frames = []
for img in img_files:
    dst = TEST_FRAMES_DIR / img.name
    try:
        dst.symlink_to(img.resolve())
    except OSError:
        shutil.copy(img, dst)
    all_frames.append(dst)

print(f"Totale frame trovati: {len(all_frames)}")
print(f"TEST_FRAMES_DIR (consolidata): {TEST_FRAMES_DIR}")

In [ ]:
# ============================================================
# Cella 4 — prepare_gt: GT remap → spazio ID del modello (per-modello)
# ============================================================
# Riporta le label GT nello spazio ID del modello passato, cosi torchmetrics non
# ha mismatch di nc. Le classi BOSS non rilevabili dal modello sono scartate.
# Ogni modello ha la propria cartella GT: stesse immagini, ID space corretto.
def prepare_gt(model_classes, model_nc, model_to_boss, gt_dir):
    gt_root = GT_ZIP_PATH

    gt_test_dir = gt_dir / "test"
    if gt_test_dir.exists():
        shutil.rmtree(gt_test_dir)

    gt_yaml_in_path = gt_root / "data.yaml"
    if not gt_yaml_in_path.exists():
        raise FileNotFoundError(f"data.yaml non trovato in: {gt_yaml_in_path}")
    with open(gt_yaml_in_path) as f:
        gt_yaml_in = yaml.safe_load(f)
    GT_CLASSES = gt_yaml_in["names"]

    # Inverso modello→BOSS: boss_id → model_id (prima occorrenza)
    BOSS_TO_MODEL = {}
    for mid, bid in model_to_boss.items():
        BOSS_TO_MODEL.setdefault(bid, mid)

    # gt_id → model_id, passando per la classe BOSS canonica (mappa universale)
    GT_TO_MODEL, unmapped = {}, []
    for gt_id, gt_name in enumerate(GT_CLASSES):
        boss = beu.resolve_to_boss(gt_name)
        if boss is None:
            unmapped.append(gt_name); continue
        mid = BOSS_TO_MODEL.get(BOSS_CLASSES.index(boss))
        if mid is None:
            unmapped.append(gt_name); continue
        GT_TO_MODEL[gt_id] = mid
    print(f"  Mappa GT→modello: {GT_TO_MODEL}")
    print(f"  Classi GT scartate (non rilevabili): {unmapped}")

    def remap_labels(src_label_dir, dst_label_dir, id_map):
        src, dst = Path(src_label_dir), Path(dst_label_dir)
        dst.mkdir(parents=True, exist_ok=True)
        remapped, dropped = 0, 0
        for lf in src.glob("*.txt"):
            out_lines = []
            for line in lf.read_text().strip().splitlines():
                parts = line.split()
                if not parts:
                    continue
                gid = int(parts[0])
                if gid not in id_map:
                    dropped += 1; continue
                parts[0] = str(id_map[gid])
                out_lines.append(" ".join(parts)); remapped += 1
            (dst / lf.name).write_text("\n".join(out_lines))
        print(f"  Annotazioni rimappate: {remapped} | Righe scartate: {dropped}")

    gt_test_imgs   = gt_dir / "test" / "images"
    gt_test_labels = gt_dir / "test" / "labels"
    gt_test_imgs.mkdir(parents=True, exist_ok=True)

    src_imgs   = gt_root / "test" / "images"
    src_labels = gt_root / "test" / "labels"
    if not src_imgs.exists():
        raise FileNotFoundError(f"Cartella immagini test non trovata: {src_imgs}")

    total_imgs = 0
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for img in src_imgs.glob(ext):
            dst = gt_test_imgs / img.name
            try:
                dst.symlink_to(img.resolve())
            except OSError:
                shutil.copy(img, dst)
            total_imgs += 1

    if not src_labels.exists():
        raise FileNotFoundError(f"Cartella label test non trovata: {src_labels}")
    remap_labels(src_labels, gt_test_labels, GT_TO_MODEL)

    model_names_list = [model_classes[i] for i in range(model_nc)]
    gt_data_yaml = {
        "path":  str(gt_dir), "train": "", "val": "test/images", "test": "",
        "nc":    model_nc,    "names": model_names_list,
    }
    with open(gt_dir / "data_boss.yaml", "w") as f:
        yaml.dump(gt_data_yaml, f, default_flow_style=False, allow_unicode=True)

    return gt_test_imgs, gt_test_labels, total_imgs

In [ ]:
# ============================================================
# Cella 5 — evaluate_model: pipeline COMPLETA per UN modello
# ============================================================
# Esegue l'intera valutazione (inferenza → distribuzione → GT eval → metriche per
# classe → griglia campione → efficienza → dashboard → report) per il modello
# passato, scrivendo in TEST_OUTPUT/<model_name>/. Chiamata due volte (Cella 6):
# stesse condizioni, istanti diversi. Ritorna un dict con le metriche per il confronto.
def evaluate_model(model_name, weights_path):
    print(f"\n{'='*64}\n### {model_name}  ({weights_path})\n{'='*64}")

    model_root     = TEST_OUTPUT / model_name
    DIR_MODEL_OUT  = model_root / "model_output"
    DIR_GT_EVAL    = model_root / "gt_eval"
    DIR_COMPARISON = model_root / "comparison"
    DIR_MARKDOWN   = model_root / "markdown"
    DIR_EFFICIENCY = model_root / "efficiency"
    for d in (DIR_MODEL_OUT, DIR_GT_EVAL, DIR_COMPARISON, DIR_MARKDOWN, DIR_EFFICIENCY):
        d.mkdir(parents=True, exist_ok=True)

    # --- Modello ---
    model = YOLO(str(weights_path))
    MODEL_CLASSES = model.names
    MODEL_NC      = len(MODEL_CLASSES)
    MODEL_TO_BOSS = beu.build_model_to_boss(MODEL_CLASSES)

    # --- Ground Truth nello spazio ID del modello ---
    gt_dir = GT_EXTRACTED / model_name
    GT_TEST_IMGS, GT_TEST_LABELS, _ = prepare_gt(MODEL_CLASSES, MODEL_NC, MODEL_TO_BOSS, gt_dir)

    def run_yolo(img_bgr):
        """Inferenza BGR con conf=0 (curva PR completa per torchmetrics)."""
        results = model.predict(source=img_bgr, conf=0, iou=IOU_THRESHOLD,
                                imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        r = results[0]
        if r.boxes is None or len(r.boxes) == 0:
            return (np.zeros((0, 4), dtype=np.float32),
                    np.array([], dtype=np.int64), np.array([], dtype=np.float32))
        return (r.boxes.xyxy.cpu().numpy(),
                r.boxes.cls.cpu().numpy().astype(np.int64),
                r.boxes.conf.cpu().numpy())

    # ---- Inferenza su tutti i frame → df_preds + frame annotati ----
    all_predictions, PREDICT_SAVE_DIR = [], None
    inference_results = model.predict(
        source=str(TEST_FRAMES_DIR), conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
        imgsz=IMG_SIZE, device=DEVICE, save=True, save_txt=True,
        project=str(DIR_MODEL_OUT), name="annotated_frames", exist_ok=True, stream=True)
    for result in inference_results:
        if PREDICT_SAVE_DIR is None:
            PREDICT_SAVE_DIR = Path(result.save_dir)
        frame_name = Path(result.path).name
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs   = boxes.conf.cpu().numpy()
        xyxy    = boxes.xyxy.cpu().numpy()
        for cid, cf, box in zip(cls_ids, confs, xyxy):
            boss_id = MODEL_TO_BOSS.get(int(cid))
            all_predictions.append({
                "frame": frame_name, "class_id": int(cid),
                "class_name": MODEL_CLASSES[int(cid)],
                "boss_class": BOSS_CLASSES[boss_id] if boss_id is not None else None,
                "confidence": float(cf),
                "x1": box[0], "y1": box[1], "x2": box[2], "y2": box[3]})
    df_preds = pd.DataFrame(all_predictions)
    df_preds.to_csv(DIR_MODEL_OUT / "predictions.csv", index=False)
    print(f"Predizioni: {len(df_preds)} | frame con pred: "
          f"{df_preds['frame'].nunique() if not df_preds.empty else 0}")

    # ---- Distribuzione classi BOSS + confidenza ----
    if not df_preds.empty:
        df_boss = df_preds.dropna(subset=["boss_class"])
        class_counts = df_boss["boss_class"].value_counts().reindex(BOSS_CLASSES, fill_value=0)
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.bar(class_counts.index, class_counts.values, color=plt.cm.tab20.colors[:NUM_CLASSES])
        ax.set_title(f"Distribuzione classi BOSS rilevate — {model_name}", fontsize=14)
        ax.set_xlabel("Classe BOSS"); ax.set_ylabel("Numero rilevamenti")
        plt.xticks(rotation=45, ha="right"); plt.tight_layout()
        plt.savefig(DIR_MODEL_OUT / "plot_class_distribution.png", dpi=150); plt.show()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(df_preds["confidence"], bins=50, color="steelblue", edgecolor="white")
        ax.axvline(CONF_THRESHOLD, color="red", linestyle="--", label=f"soglia={CONF_THRESHOLD}")
        ax.set_title(f"Distribuzione confidenza — {model_name}", fontsize=13)
        ax.set_xlabel("Confidenza"); ax.set_ylabel("Frequenza"); ax.legend()
        plt.tight_layout()
        plt.savefig(DIR_MODEL_OUT / "plot_confidence_distribution.png", dpi=150); plt.show()

    # ---- Valutazione quantitativa (torchmetrics + TP/FP/FN greedy matching) ----
    metric_50   = MeanAveragePrecision(iou_type="bbox", class_metrics=True, iou_thresholds=[0.5])
    metric_full = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
    tp_pc = np.zeros(MODEL_NC, dtype=np.int64)
    fp_pc = np.zeros(MODEL_NC, dtype=np.int64)
    fn_pc = np.zeros(MODEL_NC, dtype=np.int64)

    processed = 0
    for label_path in tqdm(sorted(GT_TEST_LABELS.glob("*.txt")), desc=f"Eval GT ({model_name})"):
        img_name = label_path.stem
        img_path = None
        for ext in (".jpg", ".jpeg", ".png"):
            p = GT_TEST_IMGS / (img_name + ext)
            if p.exists():
                img_path = p; break
        if img_path is None:
            continue
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]

        gt_boxes, gt_labels = [], []
        for line in label_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            sid = int(parts[0]); coords = [float(v) for v in parts[1:]]
            if len(coords) == 4:
                cx, cy, bw, bh = coords
                x1 = (cx - bw/2)*w; y1 = (cy - bh/2)*h
                x2 = (cx + bw/2)*w; y2 = (cy + bh/2)*h
            else:
                xs = coords[0::2]; ys = coords[1::2]
                x1 = min(xs)*w; y1 = min(ys)*h; x2 = max(xs)*w; y2 = max(ys)*h
            gt_boxes.append([x1, y1, x2, y2]); gt_labels.append(sid)
        if not gt_boxes:
            continue

        targets = [{"boxes": torch.tensor(gt_boxes, dtype=torch.float32),
                    "labels": torch.tensor(gt_labels, dtype=torch.int64)}]
        boxes_xyxy, class_ids, scores = run_yolo(img_bgr)
        preds = [{"boxes": torch.tensor(boxes_xyxy, dtype=torch.float32),
                  "scores": torch.tensor(scores, dtype=torch.float32),
                  "labels": torch.tensor(class_ids, dtype=torch.int64)}]
        metric_50.update(preds, targets)
        metric_full.update(preds, targets)

        keep = scores >= CONF_THRESHOLD
        beu.accumulate_matches(tp_pc, fp_pc, fn_pc,
            boxes_xyxy[keep], class_ids[keep], scores[keep],
            gt_boxes, gt_labels, MATCH_IOU)
        processed += 1

    res_50 = metric_50.compute(); res_full = metric_full.compute()
    map50 = float(res_50["map"]); map5095 = float(res_full["map"])
    classes_t = res_50["classes"]; ap50_t = res_50["map_per_class"]; ap5095_t = res_full["map_per_class"]
    valid_cls = (ap50_t >= 0) & (ap5095_t >= 0)
    gt_class_indices    = classes_t[valid_cls].numpy().astype(int)
    gt_per_class_ap50   = ap50_t[valid_cls].numpy()
    gt_per_class_ap5095 = ap5095_t[valid_cls].numpy()
    prec_pc, rec_pc = beu.precision_recall_per_class(tp_pc, fp_pc, fn_pc)
    gt_per_class_p = prec_pc[gt_class_indices]; gt_per_class_r = rec_pc[gt_class_indices]
    precision = float(gt_per_class_p.mean()) if len(gt_per_class_p) else 0.0
    recall    = float(gt_per_class_r.mean()) if len(gt_per_class_r) else 0.0
    f1_score  = 2*precision*recall/(precision+recall+1e-9)
    print(f"mAP@0.5={map50:.4f} | mAP@0.5:0.95={map5095:.4f} | "
          f"P={precision:.4f} | R={recall:.4f} | F1={f1_score:.4f} | img={processed}")

    # ---- Metriche per classe + grafici ----
    def _boss_label(model_id):
        bid = MODEL_TO_BOSS.get(int(model_id))
        return BOSS_CLASSES[bid] if bid is not None else MODEL_CLASSES.get(int(model_id), str(model_id))

    gt_class_names  = [_boss_label(i) for i in gt_class_indices]
    gt_per_class_f1 = 2*(gt_per_class_p*gt_per_class_r)/(gt_per_class_p+gt_per_class_r+1e-9)
    df_metrics = pd.DataFrame({
        "Classe": gt_class_names, "AP@0.5": gt_per_class_ap50,
        "AP@0.5:0.95": gt_per_class_ap5095, "Precision": gt_per_class_p,
        "Recall": gt_per_class_r, "F1": gt_per_class_f1})
    summary_row = pd.DataFrame([{
        "Classe": "MEDIA", "AP@0.5": df_metrics["AP@0.5"].mean(),
        "AP@0.5:0.95": df_metrics["AP@0.5:0.95"].mean(),
        "Precision": df_metrics["Precision"].mean(), "Recall": df_metrics["Recall"].mean(),
        "F1": df_metrics["F1"].mean()}])
    df_metrics = pd.concat([df_metrics, summary_row], ignore_index=True)
    df_metrics.to_csv(DIR_GT_EVAL / "metrics_per_class_gt.csv", index=False)
    df_plot = df_metrics[df_metrics["Classe"] != "MEDIA"].copy()

    x = np.arange(len(df_plot)); width = 0.35
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(x - width/2, df_plot["AP@0.5"],      width, label="AP@0.5",      color="steelblue")
    ax.bar(x + width/2, df_plot["AP@0.5:0.95"], width, label="AP@0.5:0.95", color="coral")
    ax.set_title(f"AP per classe — GT ({model_name})", fontsize=13)
    ax.set_xticks(x); ax.set_xticklabels(df_plot["Classe"], rotation=45, ha="right")
    ax.set_ylabel("AP"); ax.set_ylim(0, 1.1); ax.legend(); plt.tight_layout()
    plt.savefig(DIR_GT_EVAL / "plot_ap_per_class_gt.png", dpi=150); plt.show()

    # ---- Griglia campione frame annotati ----
    N_SAMPLES = 9
    annotated_imgs = []
    if PREDICT_SAVE_DIR is not None and PREDICT_SAVE_DIR.exists():
        annotated_imgs = list(PREDICT_SAVE_DIR.glob("*.jpg")) + list(PREDICT_SAVE_DIR.glob("*.png"))
    if annotated_imgs:
        sample = random.sample(annotated_imgs, min(N_SAMPLES, len(annotated_imgs)))
        grid_size = int(np.ceil(np.sqrt(len(sample))))
        fig, axes = plt.subplots(grid_size, grid_size, figsize=(5*grid_size, 4*grid_size))
        axes_flat = axes.flat if hasattr(axes, "flat") else [axes]
        for ax, img_path in zip(axes_flat, sample):
            img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
            ax.imshow(img); ax.set_title(img_path.name, fontsize=7); ax.axis("off")
        for ax in list(axes_flat)[len(sample):]:
            ax.axis("off")
        plt.suptitle(f"Campione frame — predizioni {model_name}", fontsize=14)
        plt.tight_layout()
        plt.savefig(DIR_MODEL_OUT / "plot_sample_predictions.png", dpi=150); plt.show()

    # ---- Efficienza: GMACs / GFLOPs / parametri su grafo ONNX ----
    eff = None
    try:
        export_pt = DIR_EFFICIENCY / Path(str(weights_path)).name
        shutil.copy(weights_path, export_pt)
        export_model = YOLO(str(export_pt))
        ONNX_PATH = Path(export_model.export(format="onnx", imgsz=IMG_SIZE,
                          opset=13, dynamic=False, simplify=False))
        eff = beu.count_onnx_flops_params(ONNX_PATH)
        pd.DataFrame([{
            "model": model_name, "onnx": ONNX_PATH.name,
            "input_h": IMG_SIZE, "input_w": IMG_SIZE,
            "params": eff["params"], "params_M": round(eff["params"]/1e6, 4),
            "gmacs": round(eff["gmacs"], 4), "gflops": round(eff["gflops"], 4),
        }]).to_csv(DIR_EFFICIENCY / "flops_params.csv", index=False)
        print(f"GMACs={eff['gmacs']:.3f} | params={eff['params']/1e6:.3f} M")
    except Exception as e:
        print(f"[efficienza] GMACs non calcolati: {e}")

    # ---- Dashboard riepilogo (GT vs modello) ----
    gt_img_count = len(list(GT_TEST_IMGS.glob("*.jpg"))) + len(list(GT_TEST_IMGS.glob("*.png")))
    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"B.O.S.S. — Riepilogo {model_name} su recordings", fontsize=16, fontweight="bold")

    ax1 = fig.add_subplot(2, 3, 1)
    metrics_summary = {"mAP@0.5": map50, "mAP@0.5:0.95": map5095,
                       "Precision": precision, "Recall": recall, "F1 Score": f1_score}
    bars_h = ax1.barh(list(metrics_summary.keys()), list(metrics_summary.values()),
                      color=["#2196F3", "#1976D2", "#4CAF50", "#FF9800", "#9C27B0"])
    for bar, val in zip(bars_h, metrics_summary.values()):
        ax1.text(min(val+0.02, 0.95), bar.get_y()+bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=10, fontweight="bold")
    ax1.set_xlim(0, 1.1); ax1.set_title("Metriche Aggregate vs GT", fontsize=11)
    ax1.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="soglia 0.5"); ax1.legend(fontsize=8)

    ax2 = fig.add_subplot(2, 3, 2)
    ax2.barh(df_plot["Classe"], df_plot["AP@0.5"], color="steelblue")
    ax2.set_title("AP@0.5 per Classe", fontsize=11); ax2.set_xlim(0, 1)
    ax2.axvline(map50, color="red", linestyle="--", alpha=0.7, label=f"media={map50:.2f}"); ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(2, 3, 3)
    ax3.barh(df_plot["Classe"], df_plot["F1"], color="coral")
    ax3.set_title("F1 Score per Classe", fontsize=11); ax3.set_xlim(0, 1)
    ax3.axvline(f1_score, color="red", linestyle="--", alpha=0.7, label=f"media={f1_score:.2f}"); ax3.legend(fontsize=8)

    ax4 = fig.add_subplot(2, 3, 4)
    scatter_colors = plt.cm.tab20.colors[:len(df_plot)]
    for i, (_, row) in enumerate(df_plot.iterrows()):
        ax4.scatter(row["Recall"], row["Precision"], color=scatter_colors[i % len(scatter_colors)], s=100, zorder=5)
        ax4.annotate(row["Classe"], (row["Recall"], row["Precision"]),
                     textcoords="offset points", xytext=(5, 3), fontsize=7)
    ax4.set_xlabel("Recall"); ax4.set_ylabel("Precision")
    ax4.set_title("Precision vs Recall per Classe", fontsize=11)
    ax4.set_xlim(0, 1.05); ax4.set_ylim(0, 1.05); ax4.grid(True, alpha=0.3)

    ax5 = fig.add_subplot(2, 3, 5)
    df_boss5 = df_preds.dropna(subset=["boss_class"]) if not df_preds.empty else df_preds
    if not df_preds.empty and not df_boss5.empty:
        cc = df_boss5["boss_class"].value_counts()
        ax5.pie(cc.values, labels=cc.index, autopct="%1.1f%%", startangle=140, textprops={"fontsize": 7})
        ax5.set_title(f"Distribuzione classi BOSS — {model_name}", fontsize=11)
    else:
        ax5.text(0.5, 0.5, "Nessuna predizione mappata", ha="center", va="center"); ax5.axis("off")

    ax6 = fig.add_subplot(2, 3, 6); ax6.axis("off")
    gmacs_str  = f"{eff['gmacs']:.2f}" if eff else "n/d"
    params_str = f"{eff['params']/1e6:.2f} M" if eff else "n/d"
    info_text = (
        f"Modello:        {model_name}\n"
        f"Pesi:           {Path(str(weights_path)).name}\n"
        f"Classi modello: {MODEL_NC}\n"
        f"Classi BOSS:    {NUM_CLASSES}\n"
        f"Immagine:       {IMG_SIZE}x{IMG_SIZE}\n"
        f"Conf threshold: {CONF_THRESHOLD}\n"
        f"IoU threshold:  {IOU_THRESHOLD}\n"
        f"Device:         {DEVICE}\n"
        f"GMACs (ONNX):   {gmacs_str}\n"
        f"Parametri:      {params_str}\n\n"
        f"Frame recordings:  {len(all_frames)}\n"
        f"Pred. totali:      {len(df_preds)}\n"
        f"GT immagini test:  {gt_img_count}\n")
    ax6.text(0.05, 0.95, info_text, transform=ax6.transAxes, fontsize=10,
             verticalalignment="top", fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
    ax6.set_title("Configurazione", fontsize=11)
    plt.tight_layout()
    plt.savefig(DIR_COMPARISON / "plot_dashboard_riepilogo.png", dpi=150, bbox_inches="tight"); plt.show()

    # ---- Report Markdown (modulo condiviso, titolo parametrico) ----
    if not df_preds.empty:
        class_dist = df_preds.dropna(subset=["boss_class"])["boss_class"].value_counts().reindex(BOSS_CLASSES, fill_value=0)
        pred_total = len(df_preds); frames_with_pred = df_preds["frame"].nunique()
    else:
        class_dist = None; pred_total = 0; frames_with_pred = 0
    config = {
        "Modello": model_name, "Pesi": Path(str(weights_path)).name,
        "Classi modello": MODEL_NC, "Classi BOSS": NUM_CLASSES,
        "Dimensione immagine": f"{IMG_SIZE}x{IMG_SIZE}",
        "Conf threshold (inferenza)": CONF_THRESHOLD, "IoU threshold (NMS)": IOU_THRESHOLD,
        "Match IoU (TP/FP/FN)": MATCH_IOU, "Device": DEVICE,
        "GMACs (ONNX)": (round(eff["gmacs"], 3) if eff else "n/d"),
        "Parametri (M)": (round(eff["params"]/1e6, 3) if eff else "n/d"),
        "Frame recordings": len(all_frames), "Immagini GT (test)": gt_img_count,
    }
    metrics = {"map50": map50, "map5095": map5095, "precision": precision,
               "recall": recall, "f1": f1_score, "match_iou": MATCH_IOU,
               "conf_threshold": CONF_THRESHOLD}
    report = beu.build_report(model_name, config, metrics, df_metrics,
        class_distribution=class_dist, pred_total=pred_total,
        frames_with_pred=frames_with_pred, n_frames=len(all_frames))
    (DIR_MARKDOWN / "report.md").write_text(report, encoding="utf-8")
    (DIR_MARKDOWN / "metrics_per_class.md").write_text(
        f"# Metriche per classe - Ground Truth ({model_name})\n\n" + beu.df_to_md(df_metrics) + "\n",
        encoding="utf-8")

    return {
        "name": model_name, "weights": Path(str(weights_path)).name,
        "map50": map50, "map5095": map5095, "precision": precision,
        "recall": recall, "f1": f1_score,
        "gmacs": (eff["gmacs"] if eff else None),
        "params_M": (eff["params"]/1e6 if eff else None),
        "df_metrics": df_metrics, "root": model_root,
    }

In [ ]:
# ============================================================
# Cella 6 — Esecuzione pipeline per ENTRAMBI i modelli
# ============================================================
# Istanti diversi (uno dopo l'altro) ma stesse condizioni: gli stessi frame,
# la stessa GT sorgente, le stesse soglie e la stessa risoluzione. Ogni modello
# produce il suo dashboard/report/metriche in TEST_OUTPUT/<nome>/.
results = [evaluate_model(m["name"], m["weights"]) for m in MODELS]
print("\nModelli valutati:", [r["name"] for r in results])

In [ ]:
# ============================================================
# Cella 7 — CONFRONTO modelli: tabella + grafici (a parità di condizioni)
# ============================================================
DIR_CONFRONTO = TEST_OUTPUT / "_confronto"
DIR_CONFRONTO.mkdir(parents=True, exist_ok=True)

palette     = ["#2196F3", "#FF7043", "#66BB6A", "#AB47BC"]
agg_keys    = ["map50", "map5095", "precision", "recall", "f1"]
agg_labels  = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1"]

# Tabella di confronto affiancata
df_cmp = pd.DataFrame([{
    "Modello":       r["name"],
    "Pesi":          r["weights"],
    "mAP@0.5":       r["map50"],
    "mAP@0.5:0.95":  r["map5095"],
    "Precision":     r["precision"],
    "Recall":        r["recall"],
    "F1":            r["f1"],
    "GMACs":         r["gmacs"],
    "Params (M)":    r["params_M"],
    "mAP@0.5/GMACs": (r["map50"] / r["gmacs"] if r["gmacs"] else None),
} for r in results])
df_cmp.to_csv(DIR_CONFRONTO / "confronto_modelli.csv", index=False)
print("=== Confronto modelli (stesse condizioni) ===")
print(df_cmp.to_string(index=False))

# Grafico 1: metriche aggregate a confronto (barre raggruppate)
x = np.arange(len(agg_labels)); width = 0.8 / len(results)
fig, ax = plt.subplots(figsize=(12, 6))
for i, r in enumerate(results):
    vals = [r[k] for k in agg_keys]
    bars = ax.bar(x + (i - (len(results)-1)/2)*width, vals, width,
                  label=r["name"], color=palette[i % len(palette)])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.2f}",
                ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(agg_labels)
ax.set_ylim(0, 1.1); ax.set_ylabel("Valore")
ax.set_title("Metriche aggregate — confronto modelli", fontsize=13); ax.legend()
plt.tight_layout(); plt.savefig(DIR_CONFRONTO / "plot_confronto_aggregate.png", dpi=150); plt.show()

# Grafico 2: AP@0.5 per classe a confronto (merge sulle classi)
per_class = None
for r in results:
    d = r["df_metrics"]
    d = d[d["Classe"] != "MEDIA"][["Classe", "AP@0.5"]].rename(columns={"AP@0.5": r["name"]})
    per_class = d if per_class is None else per_class.merge(d, on="Classe", how="outer")
per_class = per_class.fillna(0.0).sort_values("Classe").reset_index(drop=True)
xc = np.arange(len(per_class)); width = 0.8 / len(results)
fig, ax = plt.subplots(figsize=(15, 6))
for i, r in enumerate(results):
    ax.bar(xc + (i - (len(results)-1)/2)*width, per_class[r["name"]], width,
           label=r["name"], color=palette[i % len(palette)])
ax.set_xticks(xc); ax.set_xticklabels(per_class["Classe"], rotation=45, ha="right")
ax.set_ylim(0, 1.1); ax.set_ylabel("AP@0.5")
ax.set_title("AP@0.5 per classe — confronto modelli", fontsize=13); ax.legend()
plt.tight_layout(); plt.savefig(DIR_CONFRONTO / "plot_confronto_ap_per_classe.png", dpi=150); plt.show()

In [ ]:
# ============================================================
# Cella 8 — Dashboard di confronto + report Markdown
# ============================================================
fig = plt.figure(figsize=(16, 6))
fig.suptitle("B.O.S.S. — Confronto YOLO11n vs YOLO11s (stesse condizioni)",
             fontsize=15, fontweight="bold")

# Pannello 1: barre raggruppate metriche aggregate
axc1 = fig.add_subplot(1, 2, 1)
xg = np.arange(len(agg_labels)); wd = 0.8 / len(results)
for i, r in enumerate(results):
    vals = [r[k] for k in agg_keys]
    axc1.bar(xg + (i - (len(results)-1)/2)*wd, vals, wd, label=r["name"],
             color=palette[i % len(palette)])
axc1.set_xticks(xg); axc1.set_xticklabels(agg_labels, rotation=20)
axc1.set_ylim(0, 1.1); axc1.set_title("Metriche aggregate", fontsize=11); axc1.legend(fontsize=8)

# Pannello 2: efficienza — mAP@0.5 vs GMACs (task s1-2: miglior mAP/GMACs)
axc2 = fig.add_subplot(1, 2, 2)
if all(r["gmacs"] for r in results):
    for i, r in enumerate(results):
        axc2.scatter(r["gmacs"], r["map50"], s=160, color=palette[i % len(palette)], zorder=5)
        axc2.annotate(f"{r['name']}\n{r['map50']:.3f} mAP / {r['gmacs']:.2f} G",
                      (r["gmacs"], r["map50"]), textcoords="offset points",
                      xytext=(8, 4), fontsize=9)
    axc2.set_xlabel("GMACs (ONNX, 640x640)"); axc2.set_ylabel("mAP@0.5")
    axc2.set_title("Efficienza: mAP@0.5 vs GMACs", fontsize=11); axc2.grid(True, alpha=0.3)
else:
    axc2.text(0.5, 0.5, "GMACs non disponibili\n(efficienza n/d)", ha="center", va="center")
    axc2.axis("off")
plt.tight_layout()
plt.savefig(DIR_CONFRONTO / "plot_dashboard_confronto.png", dpi=150, bbox_inches="tight"); plt.show()

# Report Markdown di confronto
def _md(v, nd=4):
    return f"{v:.{nd}f}" if isinstance(v, (int, float)) and v is not None else "n/d"

lines = [
    "# B.O.S.S. — Confronto modelli YOLO11 (recordings, stesse condizioni)\n",
    f"- Frame recordings: **{len(all_frames)}**",
    f"- Conf threshold: **{CONF_THRESHOLD}** | IoU NMS: **{IOU_THRESHOLD}** | Match IoU: **{MATCH_IOU}**",
    f"- Immagine: **{IMG_SIZE}x{IMG_SIZE}** | Device: **{DEVICE}**\n",
    "## Metriche aggregate\n",
    "| Modello | mAP@0.5 | mAP@0.5:0.95 | Precision | Recall | F1 | GMACs | Params (M) | mAP@0.5/GMACs |",
    "|---|---|---|---|---|---|---|---|---|",
]
for r in results:
    ratio = (r["map50"] / r["gmacs"]) if r["gmacs"] else None
    lines.append(
        f"| {r['name']} | {_md(r['map50'])} | {_md(r['map5095'])} | {_md(r['precision'])} | "
        f"{_md(r['recall'])} | {_md(r['f1'])} | {_md(r['gmacs'], 3)} | {_md(r['params_M'], 3)} | {_md(ratio, 4)} |")

best_map = max(results, key=lambda r: r["map50"])
lines.append(f"\n**mAP@0.5 migliore:** {best_map['name']} ({best_map['map50']:.4f})")
if all(r["gmacs"] for r in results):
    best_eff = max(results, key=lambda r: r["map50"] / r["gmacs"])
    lines.append(f"**Miglior rapporto mAP@0.5/GMACs:** {best_eff['name']} "
                 f"({best_eff['map50']/best_eff['gmacs']:.4f})")

(DIR_CONFRONTO / "confronto_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"\nConfronto salvato in: {DIR_CONFRONTO}")